# UniLinalg — Python quickstart

`unilinalg` is a Cython extension over the UniLinalg C ABI, shipped as a
self-contained wheel: the native library travels inside the package, so
installing it needs neither Nim nor a compiler.

```
pip install lituus-unilinalg
```

CI installs the wheel the release actually publishes and executes this
notebook against it, so a change that breaks the API breaks the build. The
outputs stored below are compared against that run too -- GitHub renders them,
so a stale one would go on being read as current. Floats are compared rounded,
because the unrefined solve just below is deliberately a couple of ULP off and
its last digits differ between machines.

## Matrix: solve a linear system

In [1]:
import unilinalg

unilinalg.version(), unilinalg.__version__

('1.1.0', '1.1.0')

`Matrix.solve` factors through LU with partial pivoting. The system
below has the hand-checkable solution (1, 2, 3).

In [2]:
a = unilinalg.Matrix.from_rows([[1.0, 2.0, 1.0],
                                [2.0, 1.0, 3.0],
                                [1.0, 1.0, 1.0]])
a.solve([8.0, 13.0, 6.0])

[1.0000000000000007, 2.0, 2.9999999999999996]

The plain solve above is 1-2 ULP off -- (1, 2, 3) is exactly
representable, but float64 Gaussian elimination doesn't land on it.
`refine=True` runs one step of UniAccurate-backed iterative refinement and
recovers the exactly-rounded answer here.

In [3]:
a.solve([8.0, 13.0, 6.0], refine=True)

[1.0, 2.0, 3.0]

A singular matrix is a domain error, not a silently wrong answer.

In [4]:
try:
    unilinalg.Matrix.from_rows([[1.0, 2.0], [2.0, 4.0]]).solve([1.0, 1.0])
except ValueError as exc:
    print("ValueError:", exc)

ValueError: solve failed: non-square, shape mismatch, or singular matrix


## Cholesky, QR, SVD

In [5]:
spd = unilinalg.Matrix.from_rows([[4.0, 2.0], [2.0, 3.0]])
lower = spd.cholesky()
lower.to_rows()

[[2.0, 0.0], [1.0, 1.4142135623730951]]

In [6]:
try:
    unilinalg.Matrix.from_rows([[1.0, 2.0], [2.0, 1.0]]).cholesky()
except ValueError as exc:
    print("ValueError:", exc)

ValueError: cholesky requires a symmetric positive-definite matrix


In [7]:
diag = unilinalg.Matrix.from_rows([[2.0, 0.0], [0.0, 5.0]])
u, s, v = diag.svd()
s

[5.0, 2.0]

## Vec2/Vec3/Vec4: fixed-dimension geometric vectors

`Vec2`/`Vec3`/`Vec4` are distinct from `Matrix`: fixed dimension, not
row/column counts. `length` (a property, not a method) routes through
UniMath's `sqrtNewtonGeneric` on the Nim side -- a real dependency, not a
decorative one (see the book).

In [8]:
x = unilinalg.Vec3(1.0, 0.0, 0.0)
y = unilinalg.Vec3(0.0, 1.0, 0.0)
x.cross(y)

Vec3(0.0, 0.0, 1.0)

In [9]:
unilinalg.Vec2(3.0, 4.0).length

5.0

In [10]:
unilinalg.Vec4(1.0, 2.0, 2.0, 0.0).length

3.0

A non-numeric component is a type error, not a coercion.

In [11]:
try:
    unilinalg.Vec2("x", 1.0)
except TypeError as exc:
    print("TypeError:", exc)

TypeError: vector components must be int or float, got str


## The C ABI underneath

The same entry points are reachable from anything that speaks C. There the
contract is expressed by NULL/negative-count/error-code returns instead of
raising -- an exception must never unwind across an ABI boundary:

```c
ulin_matrix_lu_solve(h, b, blen, out, out_cap, false);  /* negated ULIN_ERR_*
    (e.g. -ULIN_ERR_SINGULAR, -ULIN_ERR_SHAPE_MISMATCH) on failure */
ulin_matrix_cholesky(h);                         /* NULL if not SPD */
```

See `include/UniLinalg.h`, and the book for the full picture.